In [4]:
import binascii
from sage.all import *

# 1. 输入数据
# 题目给出的初始状态 (128位)
init_state = [0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0]

# 题目给出的输出状态 (256位)
output_state = [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]

ciphertext_hex = '4b3be165a0a0edd67ca8f143884826725107fd42d6a6'

# 2. 准备数据流
# 完整的 LFSR 滑动窗口数据流 = 初始状态 + 输出状态的前半部分
# 因为 output_state[0] 是由 init_state 产生的
# output_state[1] 是由 init_state[1:] + [output_state[0]] 产生的
full_stream = init_state + output_state

# 3. 构建方程组 A * Mask = B
F = GF(2)
num_vars = 128
num_eqs = 256  # 我们可以利用全部 256 个输出来构建方程，保证准确性

# 矩阵 A: 每一行是某一时刻的 128 位状态
A = Matrix(F, num_eqs, num_vars)
# 向量 B: 每一位是该时刻产生的 feedback (即 output_state)
B = vector(F, num_eqs)

print("[*] 构建方程组...")
for i in range(num_eqs):
    # 取当前时刻的窗口，长度 128
    window = full_stream[i : i + num_vars]
    A.set_row(i, window)
    
    # 结果就是对应的 output_state[i]
    B[i] = output_state[i]

# 4. 求解 Mask
print("[*] 正在求解 Mask...")
try:
    # 求解 A * x = B
    mask_vec = A.solve_right(B)
    mask = [int(x) for x in mask_vec]
    print("[+] 成功解出 Mask!")
    
    # 5. 生成 Key 并解密
    # 题目逻辑: key = bytes(int(''.join(str(bit) for bit in mask[i*8:(i+1)*8]), 2) for i in range(16))
    key_bytes = []
    for i in range(16):
        byte_bits = mask[i*8 : (i+1)*8]
        byte_val = int(''.join(str(b) for b in byte_bits), 2)
        key_bytes.append(byte_val)
    
    key = bytes(key_bytes)
    print(f"[+] Key (hex): {binascii.hexlify(key)}")
    
    # 解密 Ciphertext
    cipher = binascii.unhexlify(ciphertext_hex)
    
    # 异或解密
    # keystream 循环使用 key
    # keystream = (key * (len(plaintext)//16 + 1))[:len(plaintext)]
    keystream = (key * (len(cipher)//16 + 1))[:len(cipher)]
    
    plaintext = bytes([c ^ k for c, k in zip(cipher, keystream)])
    
    print("-" * 40)
    print(f"[+] FLAG: {plaintext}")
    print("-" * 40)

except ValueError:
    print("[-] 无解。")

[*] 构建方程组...
[*] 正在求解 Mask...
[+] 成功解出 Mask!
[+] Key (hex): b'0268a231e6db81b049faae29dd3b522d'
----------------------------------------
[+] FLAG: b'ISCTF{lf5R_jUst_So_s0}'
----------------------------------------
